# Approximate Inference: From MCMC to Variational Bayes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/approximate_inference_vi_mcmc.ipynb)

Companion notebook for the [blog post](https://sesen.ai/blog/approximate-inference-mcmc-variational-bayes). Walks through Bishop's variational linear regression (PRML §10.3), runs the same model under PyMC NUTS, and reproduces the lower-bound model selection of Bishop figure 10.9.

**Reading order**

1. Generate a noisy polynomial dataset.
2. Implement coordinate-ascent VI in pure NumPy.
3. Run NUTS on the same model with PyMC.
4. Compare predictive bands, marginal posteriors, and wall-clock cost.
5. Sweep the lower bound over polynomial degree.


## Setup


In [ ]:
# Colab: install PyMC if missing
try:
    import pymc as pm
except ImportError:
    !pip install -q pymc arviz
    import pymc as pm

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import digamma, gammaln

print(f"NumPy {np.__version__} | PyMC {pm.__version__}")


## Synthetic dataset

Thirty noisy points sampled from a degree-3 polynomial on `[-5, 5]` with noise variance 0.09. Same setup as Bishop's figure 10.9.


In [ ]:
def true_curve(x):
    return 0.10 * x ** 3 - 0.40 * x ** 2 - 0.30 * x + 1.50

NOISE_SIGMA = 0.30
BETA = 1.0 / NOISE_SIGMA ** 2
X_SCALE = 5.0

rng = np.random.default_rng(7)
x_train = np.linspace(-5.0, 5.0, 30)
t_train = true_curve(x_train) + rng.normal(0.0, NOISE_SIGMA, size=x_train.shape)
x_grid = np.linspace(-5.5, 5.5, 200)

plt.figure(figsize=(7, 3.5))
plt.plot(x_grid, true_curve(x_grid), color="#1B2D3D", lw=1.5, label="true cubic")
plt.scatter(x_train, t_train, s=22, color="#3D9B8F", zorder=4, label="data")
plt.legend(); plt.grid(alpha=0.25); plt.xlabel("x"); plt.ylabel("y");


## Polynomial features

Scaling `x` to `[-1, 1]` before raising to powers keeps the design matrix well conditioned. Without this, NUTS struggles with the geometry at degree 9.


In [ ]:
def design(x, M):
    xs = x / X_SCALE
    return np.vstack([xs ** k for k in range(M + 1)]).T

M_DEMO = 9
Phi_train = design(x_train, M_DEMO)
Phi_grid = design(x_grid, M_DEMO)
print("design matrix shape:", Phi_train.shape)
print("condition number:    %.1e" % np.linalg.cond(Phi_train))


## CAVI for Bayesian linear regression

Coordinate-ascent updates from Bishop §10.3, equations 10.94, 10.95, 10.100, 10.101, with the lower bound from equations 10.108-10.112. Forty lines.


In [ ]:
def cavi(Phi, t, beta, a0=0.0, b0=0.0, max_iter=60, tol=1e-7):
    """Coordinate-ascent VI for Bayesian linear regression with Gamma prior on alpha."""
    N, M = Phi.shape
    PhiTPhi, PhiTt, tTt = Phi.T @ Phi, Phi.T @ t, float(t @ t)

    E_alpha = 1.0
    history = {"m_N": [], "S_N": [], "E_alpha": [], "elbo": []}
    prev_elbo = -np.inf

    for _ in range(max_iter):
        # q(w) = N(m_N, S_N)        Bishop Eq 10.100, 10.101
        S_N = np.linalg.inv(E_alpha * np.eye(M) + beta * PhiTPhi)
        m_N = beta * S_N @ PhiTt
        E_wTw = m_N @ m_N + np.trace(S_N)
        # q(alpha) = Gamma(a_N, b_N)  Bishop Eq 10.94, 10.95
        a_N, b_N = a0 + M / 2, b0 + 0.5 * E_wTw
        E_alpha = a_N / b_N

        psi_a, ln_b = digamma(a_N), np.log(b_N)
        elbo = (
            0.5 * N * np.log(beta / (2 * np.pi)) - 0.5 * beta * tTt
            + beta * (m_N @ PhiTt)
            - 0.5 * beta * np.trace(PhiTPhi @ (np.outer(m_N, m_N) + S_N))
            - 0.5 * M * np.log(2 * np.pi) + 0.5 * M * (psi_a - ln_b)
            - 0.5 * (a_N / b_N) * E_wTw
            - (psi_a - ln_b) - gammaln(a_N)
            + 0.5 * np.linalg.slogdet(S_N)[1] + 0.5 * M * (1 + np.log(2 * np.pi))
            + gammaln(a_N) - (a_N - 1) * psi_a - ln_b + a_N
        )
        history["m_N"].append(m_N.copy())
        history["S_N"].append(S_N.copy())
        history["E_alpha"].append(float(E_alpha))
        history["elbo"].append(float(elbo))

        if abs(elbo - prev_elbo) < tol:
            break
        prev_elbo = elbo
    return history


def predictive(Phi_grid, m_N, S_N, beta):
    mu = Phi_grid @ m_N
    var = 1.0 / beta + np.einsum("ij,jk,ik->i", Phi_grid, S_N, Phi_grid)
    return mu, np.sqrt(var)


## Run CAVI on the degree-9 model


In [ ]:
import time

t0 = time.perf_counter()
hist = cavi(Phi_train, t_train, beta=BETA)
cavi_time = time.perf_counter() - t0

print(f"converged in {len(hist['elbo'])} iterations")
print(f"final ELBO   {hist['elbo'][-1]:.2f}")
print(f"E[alpha]     {hist['E_alpha'][-1]:.3f}")
print(f"wall-clock   {cavi_time*1000:.2f} ms")

# Predictive band
m_N, S_N = hist["m_N"][-1], hist["S_N"][-1]
mu, sd = predictive(Phi_grid, m_N, S_N, BETA)

plt.figure(figsize=(8, 4))
plt.plot(x_grid, true_curve(x_grid), color="#1B2D3D", lw=1.5, label="true curve")
plt.fill_between(x_grid, mu - 2 * sd, mu + 2 * sd, color="#D4A24C", alpha=0.30, label="±2σ")
plt.plot(x_grid, mu, color="#D4A24C", lw=2, label="predictive mean")
plt.scatter(x_train, t_train, s=22, color="#3D9B8F", zorder=4, label="data")
plt.legend(loc="lower left"); plt.grid(alpha=0.25); plt.xlabel("x"); plt.ylabel("y")
plt.title("Variational predictive distribution (degree 9 features)");


## ELBO trace

Each CAVI step provably increases the lower bound. Convergence on this model is fast.


In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(np.arange(1, len(hist["elbo"]) + 1), hist["elbo"], "o-",
         color="#3D9B8F", lw=2, ms=5)
plt.xlabel("CAVI iteration"); plt.ylabel("ELBO L(q)")
plt.grid(alpha=0.25); plt.title("Lower bound climbs and plateaus");


## Model selection via the lower bound

Sweep the polynomial degree, fit a fresh CAVI, and read off the ELBO. The true generating degree is 3.


In [ ]:
elbos = []
for M in range(1, 10):
    h = cavi(design(x_train, M), t_train, beta=BETA)
    elbos.append(h["elbo"][-1])

Ms = np.arange(1, 10)
peak = int(np.argmax(elbos))

plt.figure(figsize=(7, 3.5))
plt.plot(Ms, elbos, "o-", color="#7A2E8C", lw=2, ms=7)
plt.scatter([Ms[peak]], [elbos[peak]], s=130, facecolor="none",
            edgecolor="#D4A24C", lw=2.5, label=f"peak at M={Ms[peak]}")
plt.xticks(Ms); plt.legend(); plt.grid(alpha=0.25)
plt.xlabel("polynomial degree M"); plt.ylabel("ELBO L(q)")
plt.title("Bayesian model selection via the variational lower bound");


## MCMC comparison: PyMC NUTS

The same model under No-U-Turn-Sampler. Identical priors, same data. Coffee break required.


In [ ]:
with pm.Model() as model:
    alpha = pm.Gamma("alpha", alpha=1e-3, beta=1e-3)
    w = pm.Normal("w", mu=0.0, sigma=alpha ** -0.5, shape=M_DEMO + 1)
    mu_pm = pm.math.dot(Phi_train, w)
    pm.Normal("t_obs", mu=mu_pm, sigma=NOISE_SIGMA, observed=t_train)
    idata = pm.sample(draws=2000, tune=1000, chains=2,
                      target_accept=0.95, random_seed=11, progressbar=False)

w_samples = idata.posterior["w"].stack(sample=("chain", "draw")).values.T
print(f"NUTS drew {w_samples.shape[0]} samples from {w_samples.shape[1]} coefficients")


## Marginal posteriors: VI vs NUTS

Means agree. NUTS shows the slight tail mass that the Gaussian VI marginal underestimates.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
true_w = [1.50, -1.50, -10.0, 12.5]                     # true coeffs on x/5 basis
labels = [(1, "w_1 (linear, on x/5)"), (3, "w_3 (cubic, on x/5)")]

for ax, (k, lab) in zip(axes, labels):
    grid = np.linspace(m_N[k] - 4 * np.sqrt(S_N[k, k]),
                       m_N[k] + 4 * np.sqrt(S_N[k, k]), 300)
    vi = (1.0 / np.sqrt(2 * np.pi * S_N[k, k])) * \
         np.exp(-0.5 * (grid - m_N[k]) ** 2 / S_N[k, k])
    ax.plot(grid, vi, color="#D4A24C", lw=2.2, label="VI marginal")
    ax.hist(w_samples[:, k], bins=60, density=True,
            color="#7A2E8C", alpha=0.35, label="NUTS samples")
    ax.axvline(true_w[k], color="#1B2D3D", ls="--", lw=1.2, label="true value")
    ax.set_xlabel(lab); ax.set_ylabel("density"); ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout();


## Exercises

1. **Free `\beta`.** Replace the fixed `beta` in `cavi()` with a Gamma prior on the noise precision and add a third coordinate-ascent update for `q(\beta)`. Bishop exercise 10.26 walks through the algebra.
2. **KL direction.** Repeat the marginals comparison on a non-conjugate model (Bayesian logistic regression on a small dataset). Show that the VI Gaussian marginals are noticeably narrower than the NUTS samples and explain via the KL asymmetry.
3. **Initialisation sensitivity.** Initialise `E[alpha]` at 0.01, 1, and 100. CAVI converges to the same fixed point on this model. Re-run on a non-convex ELBO (Bayesian Gaussian mixture, Bishop §10.2) and show that the answer depends on the initialisation.
4. **Random-feature basis.** Replace the polynomial features with random Fourier features. Compare the predictive bands at the same number of features and show that random features extrapolate more sensibly outside the training range.
5. **ADVI baseline.** Add a `pm.fit(method='advi')` call alongside the NUTS run. Compare the ADVI marginals to both CAVI and NUTS. Explain when ADVI matches CAVI and when it does not.
